In [ ]:
pip install ftfy

In [ ]:
import pandas as pd
import requests
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import requests
import math
import scipy.stats as stats
import random
import datetime
import unicodedata
import re
import string
from geopy.geocoders import Nominatim
from urllib.parse import urlparse
from geopy.geocoders import Nominatim
from google.colab import files
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import time
from google.colab import files

In [ ]:
# Affichage complet colonnes, observations, etc.
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
pd.set_option('display.max_rows', None)     # (optionnel) Affiche toutes les lignes si nécessaire
pd.set_option('display.width', None)        # Pas de coupure de ligne
pd.set_option('display.max_colwidth', None) # Affiche tout le contenu des cellules

In [ ]:
# fonction
def corriger_et_nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte  # Ne traite que les chaînes

    # 1. Correction d'encodage
    try:
        texte = texte.encode('raw_unicode_escape').decode('utf-8')
    except Exception:
        try:
            texte = texte.encode('latin1', errors='ignore').decode('utf-8', errors='ignore')
        except Exception:
            pass  # On garde le texte tel quel

    # 2. Nettoyage des artefacts type Word/Excel
    texte = texte.replace('_x000D_', ' ')

    # 3. Suppression de caractères invisibles ou bizarres
    texte = re.sub(r'[\u00A0\u200B\u2028\u2029\r\n]+', ' ', texte)

    # 4. Nettoyage des espaces doubles
    texte = re.sub(r'\s+', ' ', texte)

    return texte.strip()

## Et tu l’appliques comme ceci :

## python
## df["commentaires"] = df["commentaires"].apply(corriger_et_nettoyer_texte)

In [ ]:
# Fonction traitement des faux NaN

#je crée une liste de faux nul, ce qui permettra par la suite de pouvoir les traiter
faux_nans = ['NaN', 'nan', 'None', 'NULL', 'null', 'na', '-', '--', '', ' ', '\\n']
def detecter_faux_nan(df, valeurs_fausses=faux_nans):   #on crée la fonction afin d'identifier les nans
    faux_nan_counts = {}

    for col in df.columns:
        # On travaille sur des chaînes nettoyées (strip) et converties en str
        faux = df[col].astype(str).str.strip().isin(valeurs_fausses)
        nb_faux = faux.sum()
        if nb_faux > 0:
            faux_nan_counts[col] = nb_faux

    return pd.Series(faux_nan_counts).sort_values(ascending=False)

In [ ]:
# lire la table principal commentaire
df_commentaires = pd.read_excel('/content/Principal_Commentaires.xlsx')

In [ ]:
df_commentaires.shape()

In [ ]:
# taux de valeur nul par colonnes sur la table initiale
df_commentaires.isnull().mean()*100

In [ ]:
df_commentaires_maj = df_commentaires[['id_commentaire', ## clé de la table, changer le type de int à Object
                                       'id_millesime', # clé secondaire, null à drop, changer le type à Object
                                       'id_degustateur', # pas prio, donné difficilement exploitable quand on compare avec la table liste degustateur, changer type
                                       'id_degustation', # intérressant mais pas à analyser en priorité, null qui seront à drop car vide, changer type
                                       'refus_bouchon', # Interressant pour l'analyse
                                       'refus_deviance', # Interressant pour l'analyse
                                       'refus_oxydation', # Interressant pour l'analyse
                                       'commentaire_millesime', # Interressant pour l'analyse ; on retrouve des valeurs "90-91","97-98" date ? Note ? Notes Primeurs 2019 : 95-96
                                       'commentaire_millesime_en', # à retirer ou a transformer de FR à EN
                                       'note_value', # note à passer sur 100 dans une autre colonne jusqu'à la drop (erreur avec des années dans la colonne : '2019' + une note à 101)
                                       'note_bareme', # drop les bareme à (0. ; 91. ; 89.) quand j'ai un id_mil / pas de note bareme / une note alors bareme à 20
                                       'note_5', # à drop car inutilisable en l'etat et trop de 0
                                       'note_gustatif_tannins', # non exploitable car plus de 100k de "0".
                                       'note_gustatif_alcool', # non exploitable car plus de 100k de "0".
                                       'note_gustatif_puissance', # non exploitable car plus de 100k de "0".
                                       'note_gustatif_sucrosite', # non exploitable car plus de 100k de "0".
                                       'note_gustatif_fraicheur', # non exploitable car plus de 100k de "0".
                                       'note_gustatif_longueur', # non exploitable car plus de 100k de "0".
                                       'note_gustatif_intensite', # non exploitable
                                       'note_gustatif_finesse', # non exploitable
                                       'note_gustatif_complexite', # non exploitable
                                       'apogee_debut_fm', # exploitable avec quelques correction sur la données à faire
                                       'apogee_fin_fm', # exploitable avec quelques correction sur la données à faire
                                       'parution_frosine_20', # à drop
                                       'parution_frosine_5', # à drop
                                       'annee_degustation', # utilisable
                                       'garde_text', # utilisable
                                       'validation_commentaire', # à drop, trop de 0
                                       'selection_domaine', # à drop, trop de 0
                                       'millesime_txt']] # à drop

In [ ]:
df_commentaires_maj = df_commentaires[['id_commentaire', #
                                       'id_millesime', #
                                       'id_degustation',
                                       'commentaire_millesime', #
                                       'note_value', #
                                       'note_bareme', #
                                       'apogee_debut_fm', #
                                       'apogee_fin_fm', #
                                       'annee_degustation', #
                                       'garde_text' #
                                       ]]

In [ ]:
# transformation des colonnes en STR
df_commentaires_maj['id_millesime'] = df_commentaires_maj['id_millesime'].astype(str)
df_commentaires_maj['id_commentaire'] = df_commentaires_maj['id_commentaire'].astype(str)
df_commentaires_maj['id_degustation'] = df_commentaires_maj['id_degustation'].astype(str)
df_commentaires_maj['commentaire_millesime'] = df_commentaires_maj['commentaire_millesime'].astype(str)
df_commentaires_maj['note_value'] = df_commentaires_maj['note_value'].astype(str)
df_commentaires_maj['note_bareme'] = df_commentaires_maj['note_bareme'].astype(str)
df_commentaires_maj['apogee_debut_fm'] = df_commentaires_maj['apogee_debut_fm'].astype(str)
df_commentaires_maj['apogee_fin_fm'] = df_commentaires_maj['apogee_fin_fm'].astype(str)
df_commentaires_maj['annee_degustation'] = df_commentaires_maj['annee_degustation'].astype(str)
df_commentaires_maj['garde_text'] = df_commentaires_maj['garde_text'].astype(str)

In [ ]:
# Application de la fonction anti-faux nan
df_commentaires_maj = df_commentaires_maj.apply(lambda col: col.replace(faux_nans, np.nan) if col.dtype == "object" else col)

In [ ]:
# Boucle pour corriger les erreur de texte sur les colonnes
colonne_a_convertir = ['id_commentaire', #
                        'id_millesime', #
                        'commentaire_millesime', #
                        'note_value', #
                        'note_bareme', #
                        'apogee_debut_fm', #
                        'apogee_fin_fm', #
                        'annee_degustation', #
                        'garde_text' #
                       ]
for x in colonne_a_convertir:
    df_commentaires_maj[x] = df_commentaires_maj[x].apply(corriger_et_nettoyer_texte)
# Supprimer les faux nan si il y en a après avoir changé le type en .STR
df_commentaires_maj = df_commentaires_maj.replace('nan', np.nan)

In [ ]:
#Nouveau DF
df_commentaires_maj2 = df_commentaires_maj.copy()

In [ ]:
## CORRECTION A LA MANO D'ERREURS DANS LE DF

# Droper les lignes inutiles
df_commentaires_maj2 = df_commentaires_maj2.dropna(subset=['id_millesime']) #retire les null de la colonne ID Millesime
df_commentaires_maj2 = df_commentaires_maj2.dropna(subset=['id_millesime']) #retire les null de la colonne ID Millesime
df_commentaires_maj2 = df_commentaires_maj2.drop(134104, errors = 'ignore') # retire la ligne avec un barême de 89.
df_commentaires_maj2 = df_commentaires_maj2.drop(index=[135752, 13575, 134180], errors='ignore') # retire les lignes avec un barême de 0.

# Note Value :
df_commentaires_maj2['note_value'] = df_commentaires_maj2['note_value'].str.replace(',', '.') # Convertir les "," en "."
df_commentaires_maj2.loc[148080, 'apogee_fin_fm'] = '2042'
df_commentaires_maj2.loc[148080, 'apogee_debut_fm'] = '2027'
df_commentaires_maj2.loc[148080, 'note_value'] = np.nan

df_commentaires_maj2.loc[153806, 'apogee_debut_fm'] = '2029'
df_commentaires_maj2.loc[153806, 'apogee_fin_fm'] = '2042'
df_commentaires_maj2.loc[153806, 'note_value'] = np.nan

df_commentaires_maj2.loc[154160, 'apogee_debut_fm'] = '2027'
df_commentaires_maj2.loc[154160, 'apogee_fin_fm'] = '2037'
df_commentaires_maj2.loc[154160, 'note_value'] = np.nan

df_commentaires_maj2.loc[154181, 'apogee_debut_fm'] = '2028'
df_commentaires_maj2.loc[154181, 'apogee_fin_fm'] = '2042'
df_commentaires_maj2.loc[154181, 'note_value'] = np.nan

df_commentaires_maj2.loc[136083, 'note_value'] = '100'

df_commentaires_maj2.loc[134785, 'note_value'] = np.nan

df_commentaires_maj2.loc[106686 , 'annee_degustation'] = '2018'
df_commentaires_maj2.loc[106686 , 'note_value'] = '13.5'

df_commentaires_maj2 = df_commentaires_maj2.drop(index=89742)

In [ ]:
# Nouveau DF
df_commentaires_maj3 = df_commentaires_maj2.copy()

In [ ]:
# changer le type de la colonne note_value de str à float et avant convertir les "," en "."
df_commentaires_maj3['note_value'] = df_commentaires_maj3['note_value'].str.replace(',', '.')

In [ ]:
## Boucle pour ajouter une colonne "note_value_100" avec toutes les notes avec un barême de 100
ok = 0
ko = 0
Liste = []
for x, y in zip(df_commentaires_maj3['note_value'], df_commentaires_maj3['note_bareme']):
  ## prendre en compte que'<=' not supported between instances of 'int' and 'str'
  if 0 <= float(x) <= 20 and float(y) == 20:     ## si x est entre 0 et 20 et bareme = 20
    ok += 1
    Liste.append(round(float(x)*5))
  else:
    Liste.append(float(x))
    ko += 1
print(Liste)
df_commentaires_maj3['note_value_100'] = Liste

Etudier les colonnes du dataframe commentaire

In [ ]:
df_commentaires_maj3[df_commentaires_maj3['note_value'].isin(['2029', '2028','2027','2019', '101','13.514'])]


In [ ]:
df_commentaires_maj3['note_bareme'].unique()

In [ ]:
# sauvegarder l'excel
df_commentaires_maj3.to_excel("Principal_Commentaire_Propre.xlsx", index=False)
# Télécharger sur ton PC
files.download("Principal_Commentaire_Propre.xlsx")